<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-03-prompting/lesson-3.2-structured-output/notebooks/GCP_Capstone_3.2_StructuredOutput.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2 Structured Output & JSON Mode
**Netsetos GenAI Engineering — GCP Capstone**

Pydantic schemas, response_schema, enums, few-shot examples, anyOf (discriminated unions), and a reusable `structured_output.py` module.

## Setup

In [ ]:
!pip install -q google-genai==2.21.0 pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal, Optional, Union, List
import json

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')

In [ ]:
import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


## Cell 1: Why Structured Output — Plain Text vs JSON

In [ ]:
# Unstructured: parsing this is a regex nightmare
plain = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Extract: John Doe, 32, john@ex.com, Senior Eng at Acme, $150k',
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="LOW")),
)
print('PLAIN TEXT:')
print(plain.text[:200])

# Structured: parseable, typed, validated
# No temperature / top_p / top_k: gemini-3.6-flash ignores them and every Gemini 3.x model should keep the default 1.0 (verified 2026-09-03)
class Person(BaseModel):
    name: str
    age: int
    email: str
    title: str
    salary_usd: int

structured = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Extract: John Doe, 32, john@ex.com, Senior Eng at Acme, $150k',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Person,
        thinking_config=types.ThinkingConfig(thinking_level="LOW")),
)
p: Person = draft_of(structured, Person)
print(f'\nSTRUCTURED: name={p.name} age={p.age} salary=${p.salary_usd}')

## Cell 2: Pydantic Field Descriptions Guide the Model

In [ ]:
class DocMetadata(BaseModel):
    title: str = Field(description='Document title, max 120 chars')
    summary: str = Field(description='2-sentence abstract; no marketing language')
    topics: List[str] = Field(description='3-5 distinct topics, lowercase, no duplicates')
    language: Literal['en', 'hi', 'mixed'] = Field(description='Primary language')
    has_pii: bool = Field(description='True if document contains names, emails, phone, Aadhaar, PAN')
    confidence: Literal['high', 'medium', 'low']

text = '''Q4 2025 earnings: Revenue $12.3M (+18% YoY). CEO Priya Sharma noted strong India growth.
Contact: investor@acme.in, +91-98765-43210. PAN: ABCDE1234F.'''

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Extract document metadata:\n{text}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=DocMetadata,
        thinking_config=types.ThinkingConfig(thinking_level="LOW")),
)
meta: DocMetadata = draft_of(r, DocMetadata)
print(json.dumps(meta.model_dump(), indent=2, ensure_ascii=False))

## Cell 3: Enum-Only Output (Classification)

In [ ]:
# Pattern: response_mime_type='text/x.enum' for single-label classification.
# Cheaper + faster than JSON when you only need one label.

SENTIMENT_SCHEMA = {'type': 'STRING', 'enum': ['POSITIVE', 'NEGATIVE', 'NEUTRAL']}

def classify_sentiment(text):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f'Sentiment of this review: {text}',
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema=SENTIMENT_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    return r.text

for review in [
    'Best onboarding I have ever seen at a company.',
    'Wasted three hours debugging their broken SDK.',
    'The product exists. Nothing more to say.',
]:
    print(f'  {classify_sentiment(review):<8} | {review}')

## Cell 4: Few-Shot Examples in the Prompt (Format Calibration)

In [ ]:
# When the schema is ambiguous, examples beat longer descriptions.

class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    total_inr: float
    due_date: str = Field(description='ISO-8601 YYYY-MM-DD')

FEW_SHOT_PROMPT = '''Extract invoice fields. Follow the format shown.

Example:
Input: Bill from Cloudify Tech. Ref INV-2025-891. Rs. 48,500 due by 15/01/2026.
Output: {"vendor": "Cloudify Tech", "invoice_number": "INV-2025-891", "total_inr": 48500.0, "due_date": "2026-01-15"}

Example:
Input: Acme Corp INV#442 Rs 12,300 pay by 3rd Feb 2026
Output: {"vendor": "Acme Corp", "invoice_number": "INV#442", "total_inr": 12300.0, "due_date": "2026-02-03"}

Now extract:
Input: {input}
Output:'''

inp = 'Invoice from BlueOcean Systems  Ref #BOS/2026/017  Rs 95,750/- payable 31 Mar 2026'
r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=FEW_SHOT_PROMPT.replace('{input}', inp),
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Invoice,
        thinking_config=types.ThinkingConfig(thinking_level="LOW")),
)
inv: Invoice = draft_of(r, Invoice)
print(f'{inv.vendor} | {inv.invoice_number} | Rs {inv.total_inr:,.0f} | due {inv.due_date}')

## Cell 5: Nested Schemas & Lists

In [ ]:
# THE contract - copied verbatim from deploy/shared/documind_schemas.py at build time. 4.2 and
# the rag-api service carry the identical text; that is what makes it a contract.
#
# Two moments, two classes. ModelDraft is what Gemini is ASKED FOR: it can only cite by the
# [Source N] number it saw, and it quotes the words it relied on. RAGAnswer is what a CALLER
# receives, with ids, pages and scores the model never knew - resolve() joins the two.
class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

# The context is what resolve() looks [Source N] up in. In production it is the packed chunk
# list from retrieval; here it is three hand-written passages in the same document shape.
packed = [
    {'id': 'kb#1', 'source_uri': 'gs://documind-ai-YOUR-ID-uploads/acme/formats.md', 'page_start': 1,
     'text': 'DocuMind supports PDF, DOCX, TXT, and Markdown files up to 200 MB.', 'score': 0.91},
    {'id': 'kb#2', 'source_uri': 'gs://documind-ai-YOUR-ID-uploads/acme/ingest.md', 'page_start': 1,
     'text': 'Uploads are processed via Doc AI Layout Parser v1.5 with 500-token chunks.', 'score': 0.88},
    {'id': 'kb#3', 'source_uri': 'gs://documind-ai-YOUR-ID-uploads/acme/embed.md', 'page_start': 1,
     'text': 'Embeddings use text-embedding-005 at 768 dimensions.', 'score': 0.62},
]
CONTEXT = '\n'.join(f'[Source {n}] {c["text"]}' for n, c in enumerate(packed, 1))

# Both the file types and the size limit live in [Source 1]; the model should cite it.
question = 'Which file types are supported and what is the size limit?'

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Answer only from the context. Cite by [Source N]; a quote is the clause that answers, at most twenty-five words.\n\nContext:\n{CONTEXT}\n\nQuestion: {question}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=ModelDraft,
        thinking_config=types.ThinkingConfig(thinking_level="LOW")),
)
draft: ModelDraft = draft_of(r, ModelDraft)
ans: RAGAnswer = resolve(draft, packed)      # ids, pages, scores - from the chunks, not the model
print(f'Answerable: {ans.answerable} | Confidence: {ans.confidence}')
print(f'Answer: {ans.answer}')
cited = set()
for c in ans.citations:
    print(f'  {c.chunk_id:6} {c.source_uri.rsplit("/", 1)[-1]:12} score={c.score:.2f}  "{c.quote}"')
    cited.add(c.chunk_id)
print(f'\nchunk ids cited: {sorted(cited)}')
assert 'kb#1' in cited, 'expected the formats passage - it holds both the types and the size limit'


## Cell 6: Discriminated Union (anyOf) — Multi-Intent Routing

In [ ]:
# Multi-intent routing: a plain Union[...] materialises as JSON Schema anyOf,
# which Vertex structured output accepts. Each variant carries a Literal 'action'
# tag, so Pydantic parses the response back to the right type on r.parsed.
# NOTE: do NOT use Field(discriminator='action') here -- it emits oneOf +
# discriminator, which the google-genai Schema type rejects (ValidationError).

class SearchIntent(BaseModel):
    action: Literal['search']
    query: str
    top_k: int = 10

class SummarizeIntent(BaseModel):
    action: Literal['summarize']
    document_id: str
    length: Literal['short', 'medium', 'long']

class TranslateIntent(BaseModel):
    action: Literal['translate']
    text: str
    target_language: Literal['en', 'hi', 'ta', 'bn', 'te']

class Intent(BaseModel):
    intent: Union[SearchIntent, SummarizeIntent, TranslateIntent]
    confidence: float = Field(ge=0.0, le=1.0)

utterances = [
    'Find me the last five deployment post-mortems.',
    'Give me a long summary of document doc-9f23.',
    'Translate this to Hindi: The deadline is next Monday.',
]
for u in utterances:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Classify user intent: {u}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=Intent,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    i: Intent = draft_of(r, Intent)
    print(f'{i.intent.action:<10} conf={i.confidence:.2f}  |  {u}')

## Cell 7: When Schema Fails — Guardrails & Fallback Parsing

In [ ]:
# Even with response_schema the parsed value can be None - on truncation, on a safety block,
# and on ANY validation failure. The last is the common case on a corpus of statutes: a citation
# quote longer than the contract's 200 characters (4.8, F21). draft_of() repairs that one; the
# others are reported with the model's finish reason - never as a refusal.

def safe_extract(client, prompt, schema, max_tokens=2048):
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema,
            max_output_tokens=max_tokens,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    if r.parsed is not None:
        return r.parsed, 'parsed'
    reason = getattr(getattr((r.candidates or [None])[0], 'finish_reason', None), 'name', 'NO_CANDIDATES')
    if reason == 'MAX_TOKENS' and max_tokens < 8192:
        return safe_extract(client, prompt, schema, max_tokens * 4)   # cut off: once more, with room
    try:
        return draft_of(r, schema), 'repaired'                        # a long quote, trimmed and re-validated
    except RuntimeError as e:
        return None, f'failed: {reason}: {e}'

class SimpleFact(BaseModel):
    fact: str
    source: Optional[str] = None

val, status = safe_extract(client, 'What year did the Apollo 11 mission land?', SimpleFact)
print(f'status={status} | value={val}')


## Cell 8: The structured-output module (the contract is the kit's deploy/shared/documind_schemas.py)

In [ ]:
# DocuMind structured_output module: the canonical schemas reused across Modules 4-12.
# The answer contract below is not defined here - it is COPIED from deploy/shared/
# documind_schemas.py, so this notebook, 4.2 and the rag-api service cannot disagree.

# --- the answer contract: identical to deploy/shared/documind_schemas.py ---
class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

# --- the other schemas this lesson standardises ---
class DocMetadata(BaseModel):
    title: str
    summary: str
    topics: List[str]
    language: Literal['en','hi','mixed']
    has_pii: bool

class ExtractedEntity(BaseModel):
    kind: Literal['person','org','location','date','money','pii']
    text: str
    start: int
    end: int

class ClassificationResult(BaseModel):
    label: str
    confidence: float = Field(ge=0, le=1)
    reasoning: Optional[str] = None

class EvalJudgement(BaseModel):
    relevance: int = Field(ge=1, le=5)
    faithfulness: int = Field(ge=1, le=5)
    helpfulness: int = Field(ge=1, le=5)
    rationale: str

SCHEMAS = {
    'rag': ModelDraft,       # the model fills a DRAFT; resolve() makes it a RAGAnswer
    'doc_meta': DocMetadata,
    'entities': list[ExtractedEntity],   # typing.List is rejected by the SDK's schema converter; the builtin generic is accepted
    'classify': ClassificationResult,
    'judge': EvalJudgement,
    'intent': Intent,
}

def structured(prompt, schema_key, model='gemini-3.6-flash'):
    schema = SCHEMAS[schema_key]
    r = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    return draft_of(r, schema)

# Quick test
j = structured('Rate this answer "Python is snake." for the question "What is Python?"', 'judge')
print(f'Judge: rel={j.relevance} faith={j.faithfulness} help={j.helpfulness}')
print(f'Reason: {j.rationale}')
print('\nModule ready: structured(prompt, schema_key, model)')

## ✅ Lesson 3.2 Complete!

- ✅ Pydantic `BaseModel` + `response_schema` for typed extraction
- ✅ The shared answer contract: `ModelDraft` (what the model is asked for) → `resolve()` → `RAGAnswer` with full `Citation`s — one text, reused by 4.2 and the rag-api service
- ✅ `Field(description=...)` to guide the model without bloating the prompt
- ✅ `response_mime_type='text/x.enum'` for cheap single-label classification
- ✅ Few-shot examples in the prompt when schema alone is ambiguous
- ✅ Nested schemas (`List[Citation]`) for RAG answers with sources
- ✅ Discriminated `Union[...]` (anyOf) for multi-intent routing
- ✅ `.parsed is None` guard + fallback `model_validate_json` for truncation
- ✅ The structured-output module with six canonical schemas (RAG, metadata, entities, classify, judge, intent); the answer contract copied from deploy/shared/documind_schemas.py

**Next: Lesson 3.3 — Chain-of-Thought & Model Routing**